In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.config import TARGET_COLUMN, N_SPLITS, RANDOM_STATE
from src.features.preprocessing import (
    get_feature_types,
    build_preprocessor,
)
import pandas as pd
from src.features.engineering import engineer_features
from src.evaluation.evaluate import evaluate_model, compare_models
from src.models.models import logistic_regression_model, random_forest_model, catboost_model
from src.models.train import build_pipeline
from src.tuning.random_search import random_search

In [8]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

X = train_df.drop(columns=TARGET_COLUMN)
y = train_df[TARGET_COLUMN]


X = engineer_features(X)

numeric_features, categorical_features = get_feature_types(X)

preprocessor = build_preprocessor(
    numeric_features,
    categorical_features,
)

model = catboost_model()

pipeline = build_pipeline(preprocessor, model)

param_distributions = {
    "model__depth": [4, 5, 6, 7, 8, 9, 10],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__iterations": [200, 300, 500],
}

search = random_search(
    pipeline=pipeline,
    param_distributions=param_distributions,
    X=X,
    y=y,
)


print(search.best_score_)
print(search.best_params_)

0.8066254364934677
{'model__learning_rate': 0.1, 'model__iterations': 500, 'model__depth': 7}
